# NovaMind — Cérebro Integrado + TRM (Colab T4)
**Runtime → GPU (T4).** Rode na ordem.


## 1. Subir o projeto (Drive ou upload .zip)


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
PROJ='/content/drive/MyDrive/novamind'   # <-- ajuste
assert os.path.exists(PROJ); print(PROJ)


## 2. Deps + dados ARC (formato por-tarefa, p/ o nosso sistema)


In [ ]:
!pip -q install python-chess numpy torch 2>/dev/null
import os
DATA=os.path.join(PROJ,'tmp','arc_data','data')
if not os.path.exists(os.path.join(DATA,'training')):
    os.system('git clone -q https://github.com/fchollet/ARC-AGI.git /content/ARC-AGI'); os.makedirs(DATA,exist_ok=True)
    for sub in ['training','evaluation']:
        d=os.path.join(DATA,sub)
        if not os.path.exists(d): os.symlink(f'/content/ARC-AGI/data/{sub}',d)
print('ARC training:',len(os.listdir(os.path.join(DATA,'training'))))


## 3. 🧠 CABEÇA UNIFICADA (maestro + memória + símbolo + neural/DMT) — o nosso sistema


In [ ]:
import torch; print('CUDA:',torch.cuda.is_available())
%cd $PROJ/tmp/arc_agi
!python nova_unified.py 120 2>&1 | tail -12


## 4. 🌱 Criação autônoma neural (prior + TTT, sem primitiva humana)


In [ ]:
%cd $PROJ/tmp/arc_agi
!python pure_neural2.py 2>&1 | tail -15


## 5. 🔁 TRM — Tiny Recursive Model (código OFICIAL, teto ~45%)
Comandos REAIS do README do repo. **Dados:** o builder do TRM espera o formato Kaggle combinado
(`arc-agi_training_challenges.json` etc., subsets training+evaluation+concept). **Siga o README do repo**
p/ baixar/colocar os dados em `kaggle/combined/` — é o passo do próprio TRM, não o nosso formato por-tarefa.
Treino real é LONGO mesmo downscaled; salve checkpoints no Drive.


In [ ]:
%cd /content
!git clone -q https://github.com/SamsungSAILMontreal/TinyRecursiveModels.git
%cd TinyRecursiveModels
!pip -q install -r requirements.txt 2>/dev/null || pip -q install einops tqdm omegaconf hydra-core wandb adam-atan2 2>/dev/null
print('repo clonado. Veja README p/ preparar os dados (kaggle/combined).')


In [ ]:
# DATASET (apos colocar os dados kaggle/combined conforme README do repo):
!python -m dataset.build_arc_dataset --input-file-prefix kaggle/combined/arc-agi --output-dir data/arc1concept-aug-1000 --subsets training evaluation concept --test-set-name evaluation 2>&1 | tail -20


In [ ]:
# TREINO 1-GPU (comando REAL do README, batch reduzido p/ T4):
import os; os.environ['HYDRA_FULL_ERROR']='1'
!python pretrain.py arch=trm data_paths="[data/arc1concept-aug-1000]" arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4 global_batch_size=64 +run_name=t4_run ema=True 2>&1 | tail -40


---
**Honesto:** células 3-4 = nosso sistema integrado (narrow, ~14,8%, criação autônoma ~7%).
Célula 5 (TRM) = o teto real (~45%) sem LLM — mas exige o data-prep do repo e treino longo na nuvem.
Erro anterior (`
` / nomes de config) corrigido com a interface real do README.
